# NHIS Part I: Cost-Related Barriers to Mental Health Care

The classification task focuses on adults who delayed or did not receive counseling or therapy from a mental health professional because of cost. The goal is not to diagnose mental illness or predict who needs treatment. The goal is to study access barriers using public-use survey data.

The 2023 NHIS Sample Adult file is the base dataset. The imputed income file and paradata file can be added later as supporting layers. The first step is to load the adult file, check the target variables, and create a clean binary target before making broader cleaning decisions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## File paths

The project is organized so the same relative paths work in PyCharm and Colab. The local workflow uses the `med_project` folder directly. The Colab mount code is included but commented out because local work is the main workflow right now.

In [2]:
# Colab option, leave commented out when working locally in PyCharm.
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/med_project")

# Local/PyCharm option.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)

Project root: C:\Users\Alexi\projects\med_project
Raw data folder: C:\Users\Alexi\projects\med_project\data\raw


## Load the NHIS Sample Adult file

The Sample Adult file is the base file for Part I because it contains the health, access, insurance, functioning, and demographic variables needed for the classification task. The paradata file is not loaded yet because the target needs to be checked first in the main adult file.

In [3]:
adult_path = DATA_RAW / "nhis_2023_sample_adult.csv"

adult = pd.read_csv(adult_path, low_memory=False)

print("Adult file shape:", adult.shape)
adult.head()

Adult file shape: (29522, 647)


,URBRRL,RATCAT_A,INCTCFLG_A,IMPINCFLG_A,LANGSPECR_A,LANGSOC_A,LANGDOC_A,LANGMED_A,LANGHM_A,PPSU,...,PROXYREL_A,PROXY_A,AVAIL_A,HHSTAT_A,INTV_MON,RECTYPE,IMPNUM_A,WTFA_A,HHX,POVRATTC_A
0,3,4,0,0,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,1,1,1,10,1,7371.139,H029691,1.01
1,4,8,0,0,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,1,1,1,10,1,3146.794,H028812,2.49
2,4,14,0,0,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,1,1,1,10,1,10807.558,H045277,6.73
3,4,10,0,0,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,1,1,1,10,1,4661.643,H021192,3.43
4,4,5,0,0,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,1,1,1,10,1,10929.554,H025576,1.27


## Check column names

Before cleaning, the file structure needs to be checked against the codebook. The first pass only confirms that the expected target columns and key design columns are present.

In [4]:
adult.columns[:30].tolist()

['URBRRL',
 'RATCAT_A',
 'INCTCFLG_A',
 'IMPINCFLG_A',
 'LANGSPECR_A',
 'LANGSOC_A',
 'LANGDOC_A',
 'LANGMED_A',
 'LANGHM_A',
 'PPSU',
 'PSTRAT',
 'HISPALLP_A',
 'RACEALLP_A',
 'DISAB3_A',
 'SCHDYMSSTC_A',
 'AFNOW',
 'REPWRKDYTC_A',
 'YRSINUS_A',
 'CITZNSTP_A',
 'PRTNREDUCP_A',
 'SPOUSEDUCP_A',
 'LEGMSTAT_A',
 'MARSTAT_A',
 'SASPPRACE_A',
 'SASPPHISP_A',
 'PRTNRAGETC_A',
 'SPOUSAGETC_A',
 'PRTNRWKFT_A',
 'PRTNRWRK_A',
 'SPOUSWKFT_A']

In [5]:
expected_columns = [
    "HHX",
    "WTFA_A",
    "PSTRAT",
    "PPSU",
    "REGION",
    "URBRRL",
    "MHTHDLY_A",
    "MHTHND_A",
]

missing_expected = [col for col in expected_columns if col not in adult.columns]

if missing_expected:
    print("Missing expected columns:", missing_expected)
else:
    print("All expected columns are present.")

All expected columns are present.


## Target variables

The target is based on two cost-related mental health care access questions.

- `MHTHDLY_A`: delayed counseling or therapy from a mental health professional because of cost
- `MHTHND_A`: needed counseling or therapy from a mental health professional but did not get it because of cost

The first cleaned target uses only clear yes/no responses. Refused, not ascertained, and don’t know responses are set aside for the first modeling version.

In [6]:
target_vars = ["MHTHDLY_A", "MHTHND_A"]

for col in target_vars:
    print(f"\n{col}")
    print(adult[col].value_counts(dropna=False).sort_index())


MHTHDLY_A
MHTHDLY_A
1     1585
2    27199
7       34
8      694
9       10
Name: count, dtype: int64

MHTHND_A
MHTHND_A
1     1487
2    27298
7       31
8      696
9       10
Name: count, dtype: int64


## Create the binary target

The new target is called `mental_health_cost_barrier`.

A value of `1` means the adult delayed counseling/therapy because of cost or needed counseling/therapy but did not get it because of cost.

A value of `0` means the adult answered no to both questions.

Other response codes are treated as missing for the first clean version.

In [7]:
def create_mental_health_cost_barrier(row):
    delayed = row["MHTHDLY_A"]
    unmet = row["MHTHND_A"]

    valid_yes_no = {1, 2}

    if delayed not in valid_yes_no or unmet not in valid_yes_no:
        return np.nan

    if delayed == 1 or unmet == 1:
        return 1

    if delayed == 2 and unmet == 2:
        return 0

    return np.nan


adult["mental_health_cost_barrier"] = adult.apply(
    create_mental_health_cost_barrier,
    axis=1
)

adult["mental_health_cost_barrier"].value_counts(dropna=False)

mental_health_cost_barrier
0.0    26945
1.0     1832
NaN      745
Name: count, dtype: int64

## Target balance

Class balance shows whether the positive class is common enough for modeling and whether accuracy alone would be misleading. For an access-barrier outcome, recall, precision, F1-score, AUC, and lift will likely be more useful than accuracy by itself.

In [8]:
target_counts = adult["mental_health_cost_barrier"].value_counts(dropna=False)
target_rates = adult["mental_health_cost_barrier"].value_counts(normalize=True, dropna=False)

target_summary = pd.DataFrame({
    "count": target_counts,
    "proportion": target_rates
})

target_summary

,count,proportion
mental_health_cost_barrier,,
0.0,26945,0.912709
1.0,1832,0.062055
NaN,745,0.025235


In [9]:
clean_target_adult = adult.dropna(subset=["mental_health_cost_barrier"]).copy()

print("Rows before target cleaning:", adult.shape[0])
print("Rows after target cleaning:", clean_target_adult.shape[0])
print("Rows removed because target was not yes/no:", adult.shape[0] - clean_target_adult.shape[0])

clean_target_adult["mental_health_cost_barrier"] = clean_target_adult["mental_health_cost_barrier"].astype(int)

Rows before target cleaning: 29522
Rows after target cleaning: 28777
Rows removed because target was not yes/no: 745


## Save the first target-cleaned file

This file is not the final modeling dataset. It only confirms that the target is usable and creates a stable starting point for the next cleaning step.

In [10]:
output_path = DATA_INTERIM / "nhis_adult_target_cleaned.csv"

clean_target_adult.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Saved shape:", clean_target_adult.shape)

Saved: C:\Users\Alexi\projects\med_project\data\interim\nhis_adult_target_cleaned.csv
Saved shape: (28777, 648)


## Candidate variables for the first cleaning pass

The first cleaning pass starts with variables that are directly connected to the project question: access, affordability, health need, functioning, insurance, geography, and basic demographics. The goal is to keep the first analytic file understandable rather than pulling in hundreds of columns at once.

The raw NHIS file contains reserved response codes. These codes need to be handled carefully because they do not all mean the same thing. For many yes/no or ordinal survey variables, values such as 7, 8, 9, 97, 98, and 99 represent refused, not ascertained, or don’t know responses. These should not be treated as ordinary numeric values.

In [11]:
# Core variables for the first NHIS cleaning pass.
# These are grouped by role so the keep/drop logic stays readable.

id_design_vars = [
    "HHX",
    "WTFA_A",
    "PSTRAT",
    "PPSU",
]

target_vars = [
    "MHTHDLY_A",
    "MHTHND_A",
    "mental_health_cost_barrier",
]

demographic_vars = [
    "AGEP_A",
    "SEX_A",
    "HISPALLP_A",
    "RACEALLP_A",
    "EDUCP_A",
    "REGION",
    "URBRRL",
    "PCNTADLT_A",
    "PCNTKIDS_A",
    "OVER65FLG_A",
    "MLTFAMFLG_A",
    "MAXEDUCP_A",
]

income_vars = [
    "RATCAT_A",
    "POVRATTC_A",
    "INCTCFLG_A",
    "IMPINCFLG_A",
]

health_need_vars = [
    "PHSTAT_A",
    "LSATIS4_A",
    "ANXEV_A",
    "DEPEV_A",
    "HYPEV_A",
    "CHLEV_A",
    "DIBEV_A",
    "COPDEV_A",
    "ARTHEV_A",
    "ASEV_A",
    "CANEV_A",
    "STREV_A",
    "CHDEV_A",
]

functioning_vars = [
    "DISAB3_A",
    "VISIONDF_A",
    "HEARINGDF_A",
    "DIFF_A",
    "COMDIFF_A",
    "COGMEMDFF_A",
    "UPPSLFCR_A",
    "SOCERRNDS_A",
    "SOCSCLPAR_A",
    "SOCWRKLIM_A",
]

insurance_access_vars = [
    "NOTCOV_A",
    "COVER_A",
    "COVER65_A",
    "MEDICARE_A",
    "MEDICAID_A",
    "PRIVATE_A",
    "PAYBLL12M_A",
    "PAYNOBLLNW_A",
    "PAYWORRY_A",
    "MEDDL12M_A",
    "MEDNG12M_A",
    "USUALPL_A",
    "USPLKIND_A",
    "LASTDR_A",
    "WELLVIS_A",
    "VIRAPP12M_A",
]

candidate_vars = (
    id_design_vars
    + target_vars
    + demographic_vars
    + income_vars
    + health_need_vars
    + functioning_vars
    + insurance_access_vars
)

candidate_vars = list(dict.fromkeys(candidate_vars))

missing_candidate_vars = [col for col in candidate_vars if col not in clean_target_adult.columns]
available_candidate_vars = [col for col in candidate_vars if col in clean_target_adult.columns]

print("Candidate variables requested:", len(candidate_vars))
print("Candidate variables available:", len(available_candidate_vars))
print("Candidate variables missing:", len(missing_candidate_vars))

if missing_candidate_vars:
    print("\nMissing variables:")
    print(missing_candidate_vars)

Candidate variables requested: 62
Candidate variables available: 62
Candidate variables missing: 0


## First analytic subset

The first subset keeps only the target, identifiers needed for merging or survey-aware checks, and the first set of candidate predictors. The survey weight and design variables are retained in the file but will not be used as model predictors.

In [12]:
nhis_first_pass = clean_target_adult[available_candidate_vars].copy()

print("First-pass NHIS shape:", nhis_first_pass.shape)
nhis_first_pass.head()

First-pass NHIS shape: (28777, 62)


,HHX,WTFA_A,PSTRAT,PPSU,MHTHDLY_A,MHTHND_A,mental_health_cost_barrier,AGEP_A,SEX_A,HISPALLP_A,...,PAYBLL12M_A,PAYNOBLLNW_A,PAYWORRY_A,MEDDL12M_A,MEDNG12M_A,USUALPL_A,USPLKIND_A,LASTDR_A,WELLVIS_A,VIRAPP12M_A
0,H029691,7371.139,103,2,2,2,0,67,1,3,...,2,NaN,1,2,2,1,1.0,1,1.0,2
1,H028812,3146.794,122,2,2,2,0,73,1,2,...,2,NaN,3,2,2,1,1.0,1,1.0,2
2,H045277,10807.558,122,2,2,2,0,48,1,3,...,2,NaN,3,2,2,1,1.0,1,NaN,2
3,H021192,4661.643,122,2,2,2,0,42,2,2,...,2,NaN,3,2,2,1,1.0,1,NaN,2
4,H025576,10929.554,122,2,2,2,0,50,2,2,...,2,NaN,3,2,2,1,1.0,1,NaN,2


## Missingness overview

Missingness can come from several sources: true missing data, skipped questions, or reserved response codes. The first overview checks standard missing values already stored as `NaN`. Reserved codes are handled separately after the value distributions are reviewed.

In [13]:
missing_summary = (
    nhis_first_pass
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_count"})
)

missing_summary["missing_rate"] = missing_summary["missing_count"] / len(nhis_first_pass)

missing_summary = missing_summary.sort_values("missing_rate", ascending=False)

missing_summary.head(25)

,variable,missing_count,missing_rate
53,PAYNOBLLNW_A,25889,0.899642
60,WELLVIS_A,23606,0.820308
48,COVER65_A,19307,0.670918
47,COVER_A,9470,0.329082
58,USPLKIND_A,2670,0.092782
18,MAXEDUCP_A,69,0.002398
6,mental_health_cost_barrier,0,0.000000
7,AGEP_A,0,0.000000
8,SEX_A,0,0.000000
0,HHX,0,0.000000


## Value checks for candidate variables

The next tables show the raw value counts for the candidate variables. These checks help identify sparse variables, skip-pattern variables, and reserved response codes before any recoding is applied.

In [14]:
def show_value_counts(df, columns, max_unique=20):
    for col in columns:
        if col not in df.columns:
            continue

        unique_count = df[col].nunique(dropna=False)
        print("\n" + "=" * 80)
        print(f"{col} | unique values: {unique_count}")

        if unique_count <= max_unique:
            print(df[col].value_counts(dropna=False).sort_index())
        else:
            print(df[col].describe())


review_groups = {
    "Target": target_vars,
    "Demographics": demographic_vars,
    "Income": income_vars,
    "Health need": health_need_vars,
    "Functioning": functioning_vars,
    "Insurance and access": insurance_access_vars,
}

for group_name, cols in review_groups.items():
    print("\n\n" + "#" * 100)
    print(group_name)
    print("#" * 100)
    show_value_counts(nhis_first_pass, cols)



####################################################################################################
Target
####################################################################################################

MHTHDLY_A | unique values: 2
MHTHDLY_A
1     1582
2    27195
Name: count, dtype: int64

MHTHND_A | unique values: 2
MHTHND_A
1     1486
2    27291
Name: count, dtype: int64

mental_health_cost_barrier | unique values: 2
mental_health_cost_barrier
0    26945
1     1832
Name: count, dtype: int64


####################################################################################################
Demographics
####################################################################################################

AGEP_A | unique values: 70
count    28777.000000
mean        53.241373
std         18.635762
min         18.000000
25%         37.000000
50%         55.000000
75%         69.000000
max         99.000000
Name: AGEP_A, dtype: float64

SEX_A | unique values: 4
SEX_A
1    13139


## Reserved code cleaning rules

NHIS uses reserved response codes for values such as refused, not ascertained, and don’t know. These codes cannot be cleaned with one broad rule because some variables use values like 7, 8, 9, and 10 as real response categories.

Education is the clearest example. In `EDUCP_A` and `MAXEDUCP_A`, values 7, 8, 9, and 10 are valid education levels. Only 97, 98, and 99 should be treated as missing for those variables.

The first cleaning pass uses variable-specific rules so valid categories are not accidentally removed.

In [15]:
nhis_clean_step1 = nhis_first_pass.copy()

single_digit_reserved_vars = [
    "SEX_A",
    "PHSTAT_A",
    "LSATIS4_A",
    "ANXEV_A",
    "DEPEV_A",
    "HYPEV_A",
    "CHLEV_A",
    "DIBEV_A",
    "COPDEV_A",
    "ARTHEV_A",
    "ASEV_A",
    "CANEV_A",
    "STREV_A",
    "CHDEV_A",
    "DISAB3_A",
    "VISIONDF_A",
    "HEARINGDF_A",
    "DIFF_A",
    "COMDIFF_A",
    "COGMEMDFF_A",
    "UPPSLFCR_A",
    "SOCERRNDS_A",
    "SOCSCLPAR_A",
    "SOCWRKLIM_A",
    "NOTCOV_A",
    "COVER_A",
    "COVER65_A",
    "MEDICARE_A",
    "MEDICAID_A",
    "PRIVATE_A",
    "PAYBLL12M_A",
    "PAYNOBLLNW_A",
    "PAYWORRY_A",
    "MEDDL12M_A",
    "MEDNG12M_A",
    "USUALPL_A",
    "USPLKIND_A",
    "LASTDR_A",
    "WELLVIS_A",
    "VIRAPP12M_A",
]

two_digit_reserved_vars = [
    "AGEP_A",
    "EDUCP_A",
    "MAXEDUCP_A",
    "RATCAT_A",
]

for col in single_digit_reserved_vars:
    if col in nhis_clean_step1.columns:
        nhis_clean_step1[col] = nhis_clean_step1[col].replace({
            7: np.nan,
            8: np.nan,
            9: np.nan
        })

for col in two_digit_reserved_vars:
    if col in nhis_clean_step1.columns:
        nhis_clean_step1[col] = nhis_clean_step1[col].replace({
            97: np.nan,
            98: np.nan,
            99: np.nan
        })

print("Variable-specific reserved-code cleaning complete.")

Variable-specific reserved-code cleaning complete.


## Missingness after reserved-code recoding

The updated missingness table shows how much information is lost after reserved codes are treated as missing. Variables with heavy missingness will need a keep/drop decision before modeling.

In [16]:
missing_after_reserved = (
    nhis_clean_step1
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_count"})
)

missing_after_reserved["missing_rate"] = missing_after_reserved["missing_count"] / len(nhis_clean_step1)

missing_after_reserved = missing_after_reserved.sort_values("missing_rate", ascending=False)

missing_after_reserved.head(30)

,variable,missing_count,missing_rate
53,PAYNOBLLNW_A,25938,0.901345
60,WELLVIS_A,23693,0.823331
48,COVER65_A,19338,0.671995
47,COVER_A,9470,0.329082
58,USPLKIND_A,2680,0.093130
11,EDUCP_A,130,0.004517
51,PRIVATE_A,110,0.003822
49,MEDICARE_A,88,0.003058
35,CHDEV_A,82,0.002849
46,NOTCOV_A,77,0.002676


## Initial keep/drop review table

The first review table does not make final decisions. It gives a compact view of each candidate variable, its missing rate, and a starter decision. Final decisions should use the codebook, EDA, and modeling purpose together.

In [17]:
variable_roles = {}

for col in id_design_vars:
    variable_roles[col] = "identifier/design"
for col in target_vars:
    variable_roles[col] = "target/documentation"
for col in demographic_vars:
    variable_roles[col] = "demographic"
for col in income_vars:
    variable_roles[col] = "income/poverty"
for col in health_need_vars:
    variable_roles[col] = "health need"
for col in functioning_vars:
    variable_roles[col] = "functioning/disability"
for col in insurance_access_vars:
    variable_roles[col] = "insurance/access"

review_table = missing_after_reserved.copy()
review_table["role"] = review_table["variable"].map(variable_roles).fillna("unassigned")

def starter_decision(row):
    variable = row["variable"]
    missing_rate = row["missing_rate"]

    if variable in ["HHX", "WTFA_A", "PSTRAT", "PPSU"]:
        return "keep for merge/EDA, exclude from model predictors"

    if variable in ["MHTHDLY_A", "MHTHND_A"]:
        return "keep for documentation, exclude from model predictors"

    if variable == "mental_health_cost_barrier":
        return "target"

    if missing_rate >= 0.60:
        return "review; likely drop or subgroup only"

    if missing_rate >= 0.30:
        return "review before modeling"

    return "candidate predictor"

review_table["starter_decision"] = review_table.apply(starter_decision, axis=1)

review_table = review_table[
    ["variable", "role", "missing_count", "missing_rate", "starter_decision"]
].sort_values(["starter_decision", "missing_rate"], ascending=[True, False])

review_table

,variable,role,missing_count,missing_rate,starter_decision
58,USPLKIND_A,insurance/access,2680,0.093130,candidate predictor
11,EDUCP_A,demographic,130,0.004517,candidate predictor
51,PRIVATE_A,insurance/access,110,0.003822,candidate predictor
49,MEDICARE_A,insurance/access,88,0.003058,candidate predictor
35,CHDEV_A,health need,82,0.002849,candidate predictor
...,...,...,...,...,...
47,COVER_A,insurance/access,9470,0.329082,review before modeling
53,PAYNOBLLNW_A,insurance/access,25938,0.901345,review; likely drop or subgroup only
60,WELLVIS_A,insurance/access,23693,0.823331,review; likely drop or subgroup only
48,COVER65_A,insurance/access,19338,0.671995,review; likely drop or subgroup only


## Quick check after reserved-code cleaning

Education and family education should keep valid categories 7, 8, 9, and 10. These values are real degree categories, not missing codes.

In [18]:
check_cols = ["AGEP_A", "EDUCP_A", "MAXEDUCP_A", "RATCAT_A", "POVRATTC_A"]

for col in check_cols:
    if col in nhis_clean_step1.columns:
        print("\n" + "=" * 80)
        print(col)
        if nhis_clean_step1[col].nunique(dropna=False) <= 30:
            print(nhis_clean_step1[col].value_counts(dropna=False).sort_index())
        else:
            print(nhis_clean_step1[col].describe())


AGEP_A
count    28718.000000
mean        53.151055
std         18.547925
min         18.000000
25%         37.000000
50%         55.000000
75%         68.000000
max         85.000000
Name: AGEP_A, dtype: float64

EDUCP_A
EDUCP_A
1.0     1965
2.0      485
3.0      672
4.0     6616
5.0     4248
6.0     1126
7.0     2587
8.0     6655
9.0     3204
10.0    1089
NaN      130
Name: count, dtype: int64

MAXEDUCP_A
MAXEDUCP_A
1.0     1192
2.0      353
3.0      476
4.0     5256
5.0     3898
6.0     1280
7.0     2798
8.0     7382
9.0     4392
10.0    1681
NaN       69
Name: count, dtype: int64

RATCAT_A
RATCAT_A
1      835
2      894
3     1271
4     1119
5     1490
6     1245
7     1386
8     2401
9     2496
10    1960
11    1759
12    1591
13    1382
14    8948
Name: count, dtype: int64

POVRATTC_A
count    28777.000000
mean         4.108741
std          2.957163
min          0.000000
25%          1.810000
50%          3.310000
75%          5.650000
max         11.000000
Name: POVRATTC_A, dtyp

## Save the first cleaned candidate file

The saved file is still an interim file. It contains the cleaned target and the first candidate predictors after reserved-code handling. The next pass will refine keep/drop decisions and add the imputed income and paradata layers.

In [19]:
output_path = DATA_INTERIM / "nhis_first_pass_cleaned_candidates.csv"

nhis_clean_step1.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Saved shape:", nhis_clean_step1.shape)

Saved: C:\Users\Alexi\projects\med_project\data\interim\nhis_first_pass_cleaned_candidates.csv
Saved shape: (28777, 62)


## First model-ready variable decisions

The first model-ready file should be simple enough to explain and strong enough to model. Variables with heavy skip-pattern missingness are removed for the first version unless they can be converted into a clearer derived feature.

`COVER_A` applies to adults under 65. `COVER65_A` applies to adults 65 and older. These two variables should not be used separately because each one is missing for the other age group. A combined insurance hierarchy feature is easier to explain and safer for modeling.

`USPLKIND_A` describes the type of usual place for care. Missing values are meaningful because the question depends on having a usual place for care. The missing values are kept as a separate category.

In [20]:
nhis_model_ready = nhis_clean_step1.copy()

## Combine insurance hierarchy variables

The adult insurance hierarchy is split by age group in the public-use file. A single derived feature is created so each adult has one insurance category where possible.

In [21]:
def combine_insurance_hierarchy(row):
    age = row["AGEP_A"]

    if pd.isna(age):
        return np.nan

    if age < 65:
        return row["COVER_A"]

    return row["COVER65_A"]


nhis_model_ready["insurance_hierarchy_combined"] = nhis_model_ready.apply(
    combine_insurance_hierarchy,
    axis=1
)

print(nhis_model_ready["insurance_hierarchy_combined"].value_counts(dropna=False).sort_index())

insurance_hierarchy_combined
1.0    16752
2.0     3654
3.0     4429
4.0     2966
5.0      838
6.0       48
NaN       90
Name: count, dtype: int64


## Create broader feature groups

Some variables are useful individually, while others are easier to explain as grouped indicators. The first feature engineering pass creates simple summaries for chronic conditions, mental health history, disability/functioning, and affordability pressure.

In [22]:
chronic_condition_cols = [
    "HYPEV_A",
    "CHLEV_A",
    "DIBEV_A",
    "COPDEV_A",
    "ARTHEV_A",
    "ASEV_A",
    "CANEV_A",
    "STREV_A",
    "CHDEV_A",
]

mental_health_history_cols = [
    "ANXEV_A",
    "DEPEV_A",
]

functioning_difficulty_cols = [
    "VISIONDF_A",
    "HEARINGDF_A",
    "DIFF_A",
    "COMDIFF_A",
    "COGMEMDFF_A",
    "UPPSLFCR_A",
    "SOCERRNDS_A",
    "SOCSCLPAR_A",
]

affordability_cols = [
    "PAYBLL12M_A",
    "PAYWORRY_A",
    "MEDDL12M_A",
    "MEDNG12M_A",
]

In [23]:
def count_yes_values(df, columns):
    available_cols = [col for col in columns if col in df.columns]
    return (df[available_cols] == 1).sum(axis=1)


def count_difficulty_values(df, columns):
    available_cols = [col for col in columns if col in df.columns]
    return df[available_cols].isin([2, 3, 4]).sum(axis=1)


nhis_model_ready["chronic_condition_count"] = count_yes_values(
    nhis_model_ready,
    chronic_condition_cols
)

nhis_model_ready["mental_health_history_count"] = count_yes_values(
    nhis_model_ready,
    mental_health_history_cols
)

nhis_model_ready["functioning_difficulty_count"] = count_difficulty_values(
    nhis_model_ready,
    functioning_difficulty_cols
)

nhis_model_ready["affordability_pressure_count"] = count_yes_values(
    nhis_model_ready,
    affordability_cols
)

summary_cols = [
    "chronic_condition_count",
    "mental_health_history_count",
    "functioning_difficulty_count",
    "affordability_pressure_count",
]

nhis_model_ready[summary_cols].describe()

,chronic_condition_count,mental_health_history_count,functioning_difficulty_count,affordability_pressure_count
count,28777.000000,28777.000000,28777.000000,28777.000000
mean,1.508844,0.381033,1.142996,0.348751
std,1.554138,0.700812,1.599109,0.810943
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,1.000000,0.000000
75%,2.000000,1.000000,2.000000,0.000000
max,8.000000,2.000000,8.000000,4.000000


## Handle selected categorical missing values

For the first model-ready file, missing values in selected categorical predictors are kept as explicit categories. This avoids dropping many rows and keeps skip-pattern information visible.

The target is already clean, so no target rows are removed here.

In [24]:
categorical_missing_as_unknown = [
    "USPLKIND_A",
    "insurance_hierarchy_combined",
    "EDUCP_A",
    "MAXEDUCP_A",
]

for col in categorical_missing_as_unknown:
    if col in nhis_model_ready.columns:
        nhis_model_ready[col] = nhis_model_ready[col].fillna("Unknown")

## Drop variables not used in the first model-ready file

Identifiers, survey design variables, and direct target components are removed from the model-ready file. They remain available in the interim files for merging, documentation, and weighted descriptive checks.

High-missing skip-pattern variables are also removed from the first model-ready file. They can be revisited later if a subgroup analysis is useful.

In [25]:
drop_for_model_ready = [
    "HHX",
    "WTFA_A",
    "PSTRAT",
    "PPSU",
    "MHTHDLY_A",
    "MHTHND_A",
    "COVER_A",
    "COVER65_A",
    "PAYNOBLLNW_A",
    "WELLVIS_A",
]

drop_for_model_ready = [
    col for col in drop_for_model_ready
    if col in nhis_model_ready.columns
]

nhis_model_ready = nhis_model_ready.drop(columns=drop_for_model_ready)

print("Dropped columns:", drop_for_model_ready)
print("Model-ready shape:", nhis_model_ready.shape)

Dropped columns: ['HHX', 'WTFA_A', 'PSTRAT', 'PPSU', 'MHTHDLY_A', 'MHTHND_A', 'COVER_A', 'COVER65_A', 'PAYNOBLLNW_A', 'WELLVIS_A']
Model-ready shape: (28777, 57)


## Final missingness check before saving

The first model-ready file should have no missing target values. Predictor missingness should be limited and explainable. Any remaining missing values will be handled inside the modeling pipeline using imputation.

In [26]:
final_missing = (
    nhis_model_ready
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_count"})
)

final_missing["missing_rate"] = final_missing["missing_count"] / len(nhis_model_ready)
final_missing = final_missing.sort_values("missing_rate", ascending=False)

final_missing.head(25)

,variable,missing_count,missing_rate
43,PRIVATE_A,110,0.003822
41,MEDICARE_A,88,0.003058
29,CHDEV_A,82,0.002849
40,NOTCOV_A,77,0.002676
22,CHLEV_A,74,0.002571
42,MEDICAID_A,70,0.002432
45,PAYWORRY_A,60,0.002085
18,LSATIS4_A,59,0.002050
1,AGEP_A,59,0.002050
44,PAYBLL12M_A,53,0.001842


## Save the first NHIS model-ready file

The saved file is the first modeling version for Part I. It can be revised after the imputed income and paradata files are merged, but it is strong enough for an initial baseline model.

In [27]:
output_path = DATA_PROCESSED / "nhis_model_ready_v1.csv"

nhis_model_ready.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Saved shape:", nhis_model_ready.shape)

Saved: C:\Users\Alexi\projects\med_project\data\processed\nhis_model_ready_v1.csv
Saved shape: (28777, 57)
